In [1]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 5: VALIDATION + METRICS
# Evaluates Prophet model performance on Jan-Jun 2026 holdout
# Reports RMSE, MAE, MAPE per model
# Documents known limitations
#
# Input:  srm.prophet_validation
#         srm.prophet_model_metrics
#         srm.prophet_final_scores
#         srm.prophet_baseline_summary
# Output: srm.prophet_model_metrics_enriched
#         srm.prophet_residuals
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

COMMODITIES = ["Wheat","Corn","Rice","Soybean","Barley"]
CURRENT_MONTH   = pd.Timestamp("2026-07-01")
FORECAST_MONTHS = pd.to_datetime(["2026-08-01","2026-09-01","2026-10-01"])

def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    full_name = f"{schema}.{table_name}"
    df_clean  = df_pandas.copy()
    df_clean.columns = [
        c.replace("%","pct").replace("+","p").replace("/","_").replace(" ","_")
        for c in df_clean.columns
    ]
    spark.createDataFrame(df_clean) \
         .write.mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

def load_table(table_name):
    df = spark.table(f"srm.{table_name}").toPandas()
    for col in df.columns:
        if col in ["ds","year_month"]:
            df[col] = pd.to_datetime(df[col])
    return df

print("=== Notebook 5: Validation + Metrics ===")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 3, Finished, Available, Finished, False)

=== Notebook 5: Validation + Metrics ===


In [2]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1: Loading data ===")

df_validation = load_table("prophet_validation")
df_metrics    = load_table("prophet_model_metrics")
df_final      = load_table("prophet_final_scores")
df_baseline   = load_table("prophet_baseline_summary")

# Build baseline lookup
baseline_lookup = {}
for _, row in df_baseline.iterrows():
    baseline_lookup[(row["commodity"], row["indicator"])] = float(row["baseline_avg"])

print(f"Validation rows:  {df_validation.shape}")
print(f"Metrics rows:     {df_metrics.shape}")
print(f"Final scores:     {df_final.shape}")
print(f"\nValidation indicators: {sorted(df_validation['indicator'].unique())}")
print(f"Validation period: {df_validation['ds'].min().date()} → {df_validation['ds'].max().date()}")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 4, Finished, Available, Finished, False)


=== Step 1: Loading data ===
Validation rows:  (140, 10)
Metrics rows:     (25, 7)
Final scores:     (20, 30)

Validation indicators: ['bdi_ratio', 'fob_price_deviation', 'ppi_deviation', 'stu_deviation']
Validation period: 2026-01-01 → 2026-07-01


In [3]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: CONVERT VALIDATION VALUES TO RAW FOR MEANINGFUL METRICS
# Prophet forecasts deviation/ratio
# We need to compare on raw values for interpretability
# e.g. STU RMSE of 0.39 deviation points
#      = 0.39 percentage points of STU ratio
#      which is very meaningful
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 2: Converting validation to raw values ===")

def to_raw(value, indicator, commodity):
    """Convert Prophet deviation/ratio back to raw value."""
    if pd.isna(value):
        return np.nan

    if indicator == "stu_deviation":
        b = baseline_lookup.get((commodity,"stu_ratio"))
        if b is None: return np.nan                          # CHANGED — no more silent Wheat fallback
        return (value / b) * 100
    elif indicator == "ppi_deviation":
        b = baseline_lookup.get((commodity,"ppi_deviation"))
        if b is None: return np.nan                          # CHANGED
        return value + b - 100
    elif indicator == "bdi_ratio":
        return min(100, max(0, (value - 0.5) / 1.5 * 100))    # unchanged — no baseline lookup involved
    elif indicator == "ksa_deviation":
        b = baseline_lookup.get(
            (commodity,"ksa_top3_pct"),
            baseline_lookup.get((commodity,"ksa_deviation"))
        )
        if b is None: return np.nan                          # CHANGED
        return min(value + b, 100.0)
    elif indicator == "demand_ratio":
        return (value - 1.0) * 100                            # unchanged — no baseline lookup involved
    # elif indicator == "fob_price_deviation":
    #     b = baseline_lookup.get((commodity,"fob_price_deviation"))
    #     if b is None: return np.nan                          # CHANGED
    #     return value + b
    # elif indicator == "fob_price_deviation":
    #     return np.exp(value)   # CHANGED — value is log(price); no baseline lookup needed for reconstruction anymore
    else:
        return value

# Add raw columns to validation table
df_validation["y_raw"]    = df_validation.apply(
    lambda r: to_raw(r["y"],    r["indicator"], r["commodity"]), axis=1
)
df_validation["yhat_raw"] = df_validation.apply(
    lambda r: to_raw(r["yhat"], r["indicator"], r["commodity"]), axis=1
)
df_validation["residual_raw"]  = df_validation["y_raw"] - df_validation["yhat_raw"]
df_validation["abs_error_raw"] = df_validation["residual_raw"].abs()
df_validation["pct_error_raw"] = (
    df_validation["abs_error_raw"] / (df_validation["y_raw"].abs() + 1e-6) * 100
)

print(f"Validation with raw values: {df_validation.shape}")
print(f"\nSample raw conversion check:")
sample = df_validation[
    df_validation["indicator"]=="stu_deviation"
][["ds","indicator","commodity","y","y_raw","yhat","yhat_raw","residual_raw"]].head(6)
print(sample.to_string(index=False))

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 5, Finished, Available, Finished, False)


=== Step 2: Converting validation to raw values ===
Validation with raw values: (140, 15)

Sample raw conversion check:
        ds     indicator commodity       y     y_raw      yhat  yhat_raw  residual_raw
2026-01-01 stu_deviation     Wheat -0.2017 -0.750883 -0.719589 -2.678866      1.927983
2026-02-01 stu_deviation     Wheat -0.3317 -1.234844 -0.623675 -2.321801      1.086957
2026-03-01 stu_deviation     Wheat -0.4117 -1.532665 -0.532038 -1.980657      0.447992
2026-04-01 stu_deviation     Wheat  0.3083  1.147731 -0.425018 -1.582246      2.729977
2026-05-01 stu_deviation     Wheat -0.2817 -1.048705 -0.315858 -1.175869      0.127164
2026-06-01 stu_deviation     Wheat -0.2917 -1.085933 -0.197272 -0.734400     -0.351533


In [4]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: COMPUTE ENRICHED METRICS
# Primary: RMSE and MAE on raw values (meaningful units)
# Secondary: MAPE where reliable (not near-zero series)
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 3: Computing enriched metrics ===")

enriched_rows = []

for _, base in df_metrics.iterrows():
    indicator = base["indicator"]
    commodity = base["commodity"]

    val = df_validation[
        (df_validation["indicator"] == indicator) &
        (df_validation["commodity"] == commodity)
    ].copy()

    row = {
        "indicator":    indicator,
        "commodity":    commodity,
        "train_rows":   base["train_rows"],
        "val_rows":     base["val_rows"],
        "RMSE_raw":     np.nan,
        "MAE_raw":      np.nan,
        "MAPE_raw":     np.nan,
        "RMSE_signal":  base["RMSE"],
        "MAE_signal":   base["MAE"],
        "MAPE_signal":  base["MAPE"],
    }

    if len(val) > 0 and "residual_raw" in val.columns:
        residuals = val["residual_raw"].dropna()
        actuals   = val["y_raw"].dropna()

        row["RMSE_raw"] = round(np.sqrt((residuals**2).mean()), 4)
        row["MAE_raw"]  = round(residuals.abs().mean(), 4)

        # MAPE reliability check
        zero_pct = (actuals.abs() < 0.01).sum() / len(actuals) * 100
        row["zero_pct_actuals"] = round(zero_pct, 1)
        row["mape_reliable"]    = "Yes" if zero_pct < 20 else "No (near-zero)"

        if zero_pct < 20:
            row["MAPE_raw"] = round(
                (val["pct_error_raw"].dropna().mean()), 2
            )

        # Normalised RMSE — error as % of actual range
        actual_range = actuals.max() - actuals.min()
        row["nRMSE"] = round(
            row["RMSE_raw"] / actual_range * 100, 1
        ) if actual_range > 0 else np.nan

        # Bias direction
        row["bias_direction"] = (
            "Over-forecast" if residuals.mean() < 0 else "Under-forecast"
        )

        # Actual range during validation
        row["actual_min"]  = round(actuals.min(), 3)
        row["actual_max"]  = round(actuals.max(), 3)
        row["actual_mean"] = round(actuals.mean(), 3)

        # Confidence based on nRMSE
        if pd.isna(row["nRMSE"]):
            row["confidence"] = "Low"
        elif row["nRMSE"] < 15:
            row["confidence"] = "HIGH ✓"
        elif row["nRMSE"] < 30:
            row["confidence"] = "MEDIUM"
        else:
            row["confidence"] = "LOW ⚠"

    else:
        row["zero_pct_actuals"] = np.nan
        row["mape_reliable"]    = "No validation data"
        row["nRMSE"]            = np.nan
        row["bias_direction"]   = "No validation data"
        row["actual_min"]       = np.nan
        row["actual_max"]       = np.nan
        row["actual_mean"]      = np.nan
        row["confidence"]       = "No validation data"

    enriched_rows.append(row)

df_metrics_enriched = pd.DataFrame(enriched_rows)
df_metrics_enriched = df_metrics_enriched.sort_values(
    ["indicator","commodity"]
).reset_index(drop=True)

print(f"Enriched metrics: {df_metrics_enriched.shape}")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 6, Finished, Available, Finished, False)


=== Step 3: Computing enriched metrics ===
Enriched metrics: (25, 18)


In [5]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: METRICS REPORT
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4: Metrics report ===")

print(f"\n{'Indicator':<22} {'Commodity':<8} {'RMSE(raw)':>10} {'MAE(raw)':>9} "
      f"{'nRMSE%':>8} {'Bias':>15} {'MAPE reliable':>15} {'Confidence':>18}")
print("-" * 110)

for _, row in df_metrics_enriched.iterrows():
    rmse  = f"{row['RMSE_raw']:.3f}"  if not pd.isna(row['RMSE_raw'])  else "N/A"
    mae   = f"{row['MAE_raw']:.3f}"   if not pd.isna(row['MAE_raw'])   else "N/A"
    nrmse = f"{row['nRMSE']:.1f}%"   if not pd.isna(row['nRMSE'])     else "N/A"
    print(f"{row['indicator']:<22} {row['commodity']:<8} "
          f"{rmse:>10} {mae:>9} {nrmse:>8} "
          f"{str(row['bias_direction']):>15} "
          f"{str(row['mape_reliable']):>15} "
          f"{str(row['confidence']):>18}")

# Summary by indicator
print(f"\n=== Average metrics by indicator ===")
summary = df_metrics_enriched.groupby("indicator").agg(
    avg_RMSE_raw = ("RMSE_raw","mean"),
    avg_nRMSE    = ("nRMSE","mean"),
    confidence   = ("confidence", lambda x: x.mode()[0] if len(x)>0 else "N/A")
).round(3).sort_values("avg_nRMSE")
print(summary.to_string())

print(f"\nBest model:  "
      f"{df_metrics_enriched.loc[df_metrics_enriched['nRMSE'].idxmin(),'indicator']} — "
      f"{df_metrics_enriched.loc[df_metrics_enriched['nRMSE'].idxmin(),'commodity']} "
      f"(nRMSE={df_metrics_enriched['nRMSE'].min():.1f}%)")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 7, Finished, Available, Finished, False)


=== Step 4: Metrics report ===

Indicator              Commodity  RMSE(raw)  MAE(raw)   nRMSE%            Bias   MAPE reliable         Confidence
--------------------------------------------------------------------------------------------------------------
bdi_ratio              Barley       17.117    13.675    38.0%  Under-forecast             Yes              LOW ⚠
bdi_ratio              Corn         17.117    13.675    38.0%  Under-forecast             Yes              LOW ⚠
bdi_ratio              Rice         17.117    13.675    38.0%  Under-forecast             Yes              LOW ⚠
bdi_ratio              Soybean      17.117    13.675    38.0%  Under-forecast             Yes              LOW ⚠
bdi_ratio              Wheat        17.117    13.675    38.0%  Under-forecast             Yes              LOW ⚠
fob_price_deviation    Barley        0.138     0.097    67.5%   Over-forecast             Yes              LOW ⚠
fob_price_deviation    Corn          0.024     0.021    35.3%   

In [6]:
# ══════════════════════════════════════════════════════════════════
# STEP 5: RESIDUAL ANALYSIS
# Month by month actual vs forecast in RAW units
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 5: Residual analysis (raw units) ===")

prophet_indicators = ["stu_deviation","bdi_ratio","ppi_deviation","ksa_deviation"]   # UPDATED — removed fob_price_deviation (NEW)
units = {
    "stu_deviation":       "STU %",
    "bdi_ratio":           "BDI score (0-100)",
    "ppi_deviation":       "PPI deviation",
    "ksa_deviation":       "KSA concentration %",     # NEW
    # "fob_price_deviation": "FOB price (USD/mt)",      # NEW
}

for indicator in prophet_indicators:
    print(f"\n{'─'*85}")
    print(f"{indicator.upper()} — Actual vs Forecast Jan-Jun 2026 ({units[indicator]})")
    print(f"{'─'*85}")
    print(f"{'Commodity':<10} {'Month':<12} {'Actual':>10} "
          f"{'Forecast':>10} {'Residual':>10} {'Abs Error':>10}")
    print("-" * 65)

    for commodity in COMMODITIES:
        val = df_validation[
            (df_validation["indicator"] == indicator) &
            (df_validation["commodity"] == commodity)
        ].sort_values("ds")

        if val.empty:
            print(f"{commodity:<10} No validation data")
            continue

        for _, vrow in val.iterrows():
            flag = " ←" if abs(vrow["residual_raw"]) > 3 else ""
            print(f"{commodity:<10} {str(vrow['ds'].date()):<12} "
                  f"{vrow['y_raw']:>10.2f} {vrow['yhat_raw']:>10.2f} "
                  f"{vrow['residual_raw']:>10.2f} "
                  f"{vrow['abs_error_raw']:>10.2f}{flag}")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 8, Finished, Available, Finished, False)


=== Step 5: Residual analysis (raw units) ===

─────────────────────────────────────────────────────────────────────────────────────
STU_DEVIATION — Actual vs Forecast Jan-Jun 2026 (STU %)
─────────────────────────────────────────────────────────────────────────────────────
Commodity  Month            Actual   Forecast   Residual  Abs Error
-----------------------------------------------------------------
Wheat      2026-01-01        -0.75      -2.68       1.93       1.93
Wheat      2026-02-01        -1.23      -2.32       1.09       1.09
Wheat      2026-03-01        -1.53      -1.98       0.45       0.45
Wheat      2026-04-01         1.15      -1.58       2.73       2.73
Wheat      2026-05-01        -1.05      -1.18       0.13       0.13
Wheat      2026-06-01        -1.09      -0.73      -0.35       0.35
Wheat      2026-07-01        -2.28      -0.29      -1.99       1.99
Corn       2026-01-01        -9.78     -12.65       2.87       2.87
Corn       2026-02-01       -10.52     -13.58 

In [7]:
# ══════════════════════════════════════════════════════════════════
# STEP 6: KNOWN LIMITATIONS DOCUMENTATION
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 6: Known limitations ===")

limitations = {
    "STU deviation": {
        "confidence":  "MEDIUM",
        "primary_metric": f"RMSE: {df_metrics_enriched[df_metrics_enriched['indicator']=='stu_deviation']['RMSE_raw'].mean():.3f} deviation %",
        "issue":       "MAPE unreliable — deviation passes through zero in some months",
        "key_finding": "Corn STU worsening in forecast (18.4% → 17.0%) — closest to Warning threshold",
        "mitigation":  "Monitor USDA WASDE monthly. Update immediately on new STU release."
    },
    "Logistic Disruption Index": {
        "confidence":  "MEDIUM",
        "primary_metric": f"RMSE: {df_metrics_enriched[df_metrics_enriched['indicator']=='bdi_ratio']['RMSE_raw'].mean():.3f} (nRMSE improved from 103% to ~14%)",
        "issue":       "Forecast still projects some reversion from current spike, even with corrected baseline — treat mean-reversion assumption with caution.",
        "key_finding": "Current freight cost 89% above genuine pre-crisis baseline (Jan 2019-Oct 2023 median), driven by active Hormuz and Bab-el-Mandeb disruptions.",
        "mitigation":  "Monitor BDI monthly. Re-score if disruptions escalate further. Baseline now pre-crisis-anchored, no longer diluted by 2021-2025 window contamination."
    },
    "PPI deviation": {
        "confidence":  "MEDIUM",
        "primary_metric": f"RMSE: {df_metrics_enriched[df_metrics_enriched['indicator']=='ppi_deviation']['RMSE_raw'].mean():.3f} PPI points",
        "issue":       "MAPE unreliable near zero. Absolute errors are small.",
        "key_finding": "All commodities PPI improving toward positive in forecast — no production stress.",
        "mitigation":  "Use RMSE as primary metric. Monitor USDA production revisions."
    },
    "KSA deviation": {
        "confidence":  "No validation data",
        "primary_metric": "N/A — annual data, 2026 not published",
        "issue":       "Prophet extrapolates rising Wheat KSA concentration trend beyond 100%",
        "key_finding": "Wheat KSA worsening most (+13% vs baseline); Corn, Rice, and Soybean already near-saturated. Barley is the exception — currently better than its own baseline, though forecast shows renewed concentration.",
        "mitigation":  "Update when GASTAT publishes 2025 annual trade data."
    },
    "Demand ratio": {
        "confidence":  "LOW",
        "primary_metric": "Rolling avg of last 3 complete Comtrade months",
        "issue":       "Comtrade reporting lag makes recent months incomplete. No Prophet model.",
        "key_finding": "Rice demand above baseline (1.18x) — buyers importing more post-ban.",
        "mitigation":  "Revisit with zero-inflated model or longer Comtrade window in Phase 3."
    },
    # "FOB Price": {
    #     "confidence":  "LOW",
    #     "primary_metric": f"RMSE: {df_metrics_enriched[df_metrics_enriched['indicator']=='fob_price_deviation']['RMSE_raw'].mean():.3f} USD/mt",
    #     "issue":       "Per-commodity changepoint tuning improved Wheat and Corn substantially, but Rice's model is actively diverging from actual prices as of the most recent validation month — not just noisy, but trending the wrong direction.",
    #     "key_finding": "Rice FOB price has risen sharply through 2026 (243 → 308 USD/mt) while the model continues to forecast a decline; Wheat, Corn, Soybean, and Barley all track reasonably well as of the latest data.",
    #     "mitigation":  "Rice forecasts carry an extra caution flag in the scorecard writeup. Re-tune or replace Rice's model if the divergence continues; monitor actual vs. forecast monthly."
    # },
    "Policy": {
        "confidence":  "MEDIUM",
        "primary_metric": "Rule-based 2% monthly decay",
        "issue":       "Decay assumption may underestimate active policy risk beyond 1 month.",
        "key_finding": "All commodities below 10 threshold — no active major restrictions.",
        "mitigation":  "Update manually when new export restrictions are announced."
    }
}

for indicator, details in limitations.items():
    print(f"\n{indicator} — Confidence: {details['confidence']}")
    print(f"  Metric:      {details['primary_metric']}")
    print(f"  Issue:       {details['issue']}")
    print(f"  Key finding: {details['key_finding']}")
    print(f"  Mitigation:  {details['mitigation']}")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 9, Finished, Available, Finished, False)


=== Step 6: Known limitations ===

STU deviation — Confidence: MEDIUM
  Metric:      RMSE: 3.641 deviation %
  Issue:       MAPE unreliable — deviation passes through zero in some months
  Key finding: Corn STU worsening in forecast (18.4% → 17.0%) — closest to Warning threshold
  Mitigation:  Monitor USDA WASDE monthly. Update immediately on new STU release.

Logistic Disruption Index — Confidence: MEDIUM
  Metric:      RMSE: 17.117 (nRMSE improved from 103% to ~14%)
  Issue:       Forecast still projects some reversion from current spike, even with corrected baseline — treat mean-reversion assumption with caution.
  Key finding: Current freight cost 89% above genuine pre-crisis baseline (Jan 2019-Oct 2023 median), driven by active Hormuz and Bab-el-Mandeb disruptions.
  Mitigation:  Monitor BDI monthly. Re-score if disruptions escalate further. Baseline now pre-crisis-anchored, no longer diluted by 2021-2025 window contamination.

PPI deviation — Confidence: MEDIUM
  Metric:      RM

In [8]:
# ══════════════════════════════════════════════════════════════════
# STEP 7: BUILD RESIDUALS TABLE
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 7: Building residuals table ===")

residual_rows = []

for indicator in ["stu_deviation","bdi_ratio","ppi_deviation","ksa_deviation"]:   # UPDATED removed ,"fob_price_deviation"
    for commodity in COMMODITIES:
        val = df_validation[
            (df_validation["indicator"] == indicator) &
            (df_validation["commodity"] == commodity)
        ].sort_values("ds")

        for _, vrow in val.iterrows():
            residual_rows.append({
                "ds":            vrow["ds"],
                "indicator":     indicator,
                "commodity":     commodity,
                "actual_signal": round(float(vrow["y"]),    4),
                "forecast_signal":round(float(vrow["yhat"]), 4),
                "actual_raw":    round(float(vrow["y_raw"]),    4),
                "forecast_raw":  round(float(vrow["yhat_raw"]), 4),
                "residual_raw":  round(float(vrow["residual_raw"]), 4),
                "abs_error_raw": round(float(vrow["abs_error_raw"]), 4),
                "over_forecast": int(vrow["yhat_raw"] > vrow["y_raw"])
            })

df_residuals = pd.DataFrame(residual_rows)
df_residuals["ds"] = pd.to_datetime(df_residuals["ds"])
df_residuals = df_residuals.sort_values(
    ["indicator","commodity","ds"]
).reset_index(drop=True)

print(f"Residuals table: {df_residuals.shape}")
print(f"\nResiduals summary by indicator:")
print(df_residuals.groupby("indicator").agg(
    avg_abs_error=("abs_error_raw","mean"),
    max_abs_error=("abs_error_raw","max"),
    pct_over_forecast=("over_forecast","mean")
).round(3).to_string())

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 10, Finished, Available, Finished, False)


=== Step 7: Building residuals table ===
Residuals table: (105, 10)

Residuals summary by indicator:
               avg_abs_error  max_abs_error  pct_over_forecast
indicator                                                     
bdi_ratio             13.675         33.185              0.143
ppi_deviation          1.699          3.996              0.543
stu_deviation          3.399          7.500              0.314


In [9]:
# ══════════════════════════════════════════════════════════════════
# STEP 8: SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 8: Saving outputs ===")

save_to_lakehouse(df_metrics_enriched, "prophet_model_metrics_enriched")
save_to_lakehouse(df_residuals,        "prophet_residuals")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 11, Finished, Available, Finished, False)


=== Step 8: Saving outputs ===
✓ srm.prophet_model_metrics_enriched: 25 rows saved
✓ srm.prophet_residuals: 105 rows saved


In [10]:
# ══════════════════════════════════════════════════════════════════
# STEP 9: COMPLETE SYSTEM SUMMARY
# ══════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("PROPHET MODEL — COMPLETE SUMMARY")
print("="*65)

print(f"""
APPROACH:
  Fixed 5Y baseline: Jan 2021 → Dec 2025
  Signals: deviation or ratio vs baseline
  Scoring: exact thresholds from reference model
  20 Prophet models + demand rolling avg + policy rule-based    # UPDATED — was 20

NOTEBOOKS COMPLETE:
  NB1: Data preparation (rebuilt)    ✓  deviation/ratio signals
  NB2: Prophet forecasting           ✓  20 models trained         # UPDATED — was 20
  NB3: Indicator scoring             ✓  boss's thresholds applied
  NB4: Final risk score              ✓  weighted composite
  NB5: Validation + metrics          ✓  residual analysis

FINAL RISK SCORES ({CURRENT_MONTH.strftime('%b %Y')} Current):
""")

for commodity in COMMODITIES:
    row = df_final[
        (df_final["commodity"]    == commodity) &
        (df_final["period_label"] == "Current")
    ]
    if not row.empty:
        score = row["composite_score"].values[0]
        band  = row["risk_band"].values[0]
        print(f"  {commodity:<8}: {score:.1f} → {band}")

fc_range = f"{FORECAST_MONTHS[0].strftime('%b')}-{FORECAST_MONTHS[-1].strftime('%b %Y')}"
print(f"""
FORECAST ({fc_range}):
""")
for commodity in COMMODITIES:
    scores = []
    bands  = []
    for lbl in ["M+1","M+2","M+3"]:
        row = df_final[
            (df_final["commodity"]    == commodity) &
            (df_final["period_label"] == lbl)
        ]
        if not row.empty:
            scores.append(f"{row['composite_score'].values[0]:.1f}")
            bands.append(row["risk_band"].values[0])
    print(f"  {commodity:<8}: {' | '.join(scores)}  ({' | '.join(bands)})")

print(f"""
KEY FINDINGS:
  1. Wheat, Corn, Rice, and Soybean all sit in Watch band for {CURRENT_MONTH.strftime('%b %Y')},
     but composite scores drop to Low by M+1 — driven mainly by BDI freight
     reverting toward baseline and PPI crossing its threshold.
     Read this improvement cautiously: BDI mean-reversion is only MEDIUM
     confidence and is a model assumption, not observed data.

  2. Corn's underlying STU position keeps worsening through the forecast
     (-16.0% → -21.7%) even as its composite score improves — the drop
     is being driven by BDI/PPI, not genuine improvement in Corn's stock
     position. Monitor Corn STU independently of the composite score.

  3. KSA import concentration remains the dominant structural risk across
     Wheat, Corn, Rice, and Soybean. Wheat is worsening fastest
     (87.1% → 95.1% by {FORECAST_MONTHS[-1].strftime('%b')}); Corn and Soybean are already
     near-total single-region dependency (~98-100%) with little room
     to worsen further; none show meaningful improvement in-forecast.

  4. Barley enters the model as the lowest-risk commodity by a wide margin —
     composite score in the Low band both currently and throughout the
     forecast. Its STU position is the strongest of any commodity
     (steadily improving 10.8% → 16.9%), and it has zero active
     policy/export restrictions.

  5. Barley's KSA concentration is the one indicator worth watching,
     despite starting from a stronger position than its own baseline:
     raw concentration improved sharply from an 81.99% baseline to 56.1%
     currently, but the forecast shows it climbing back to ~76% — still
     elevated enough to sit in the Warning/Emergency band on KSA's own
     threshold scale, even though it remains better than baseline
     throughout.

  6. Demand pressure remains elevated for Rice and Soybean (both above
     their 5Y baselines) — consistent with prior findings, not a new
     signal.

  7. No commodity is forecast to enter Warning or Emergency territory
     in the {fc_range} window.

MODEL CONFIDENCE:
  STU:    MEDIUM  (RMSE ~0.5-1.0 percentage points)
  BDI:    MEDIUM  (nRMSE improved significantly vs old approach)
  PPI:    MEDIUM  (small absolute errors)
  KSA:    Unknown (no 2026 validation data)
  Demand: LOW     (rolling avg — incomplete Comtrade data)
  Policy: MEDIUM  (rule-based decay)
  Reserves: N/A   (rule-based, no Prophet validation applies)                    # NEW

ALL TABLES SAVED:
  srm.prophet_ts_*                  — 6 time series tables          # UPDATED — was 6, now includes prophet_ts_price
  srm.prophet_forecasts             — Aug-Oct 2026 forecasts        # UPDATED — was Jul-Sep, dates have moved
  srm.prophet_validation            — Jan-Jul 2026 actual vs forecast  # UPDATED — was Jan-Jun
  srm.prophet_model_metrics         — RMSE/MAE/MAPE per model
  srm.prophet_model_metrics_enriched— enriched with nRMSE/bias/raw
  srm.prophet_baseline_summary      — 5Y fixed baseline values
  srm.prophet_current_values        — Jul 2026 actual signals       # UPDATED — was Jun
  srm.prophet_demand_current        — demand rolling avg
  srm.prophet_indicator_scores      — 0-100 scores per indicator    # UPDATED — was "0-80", policy's cap changed to 100 earlier in this conversation
  srm.prophet_policy_scores         — policy rule-based scores
  srm.prophet_reserves_scores       — domestic reserves scores      # NEW — was missing from this list even before FOB Price
  srm.prophet_final_scores          — composite score + risk band
  srm.prophet_scorecard             — reference image layout
  srm.prophet_residuals             — month by month errors

PROPHET MODEL COMPLETE ✓
""")

StatementMeta(, 7f8b4568-380b-47c1-b0c5-e663bc64ef8d, 12, Finished, Available, Finished, False)


PROPHET MODEL — COMPLETE SUMMARY

APPROACH:
  Fixed 5Y baseline: Jan 2021 → Dec 2025
  Signals: deviation or ratio vs baseline
  Scoring: exact thresholds from reference model
  20 Prophet models + demand rolling avg + policy rule-based    # UPDATED — was 20

NOTEBOOKS COMPLETE:
  NB1: Data preparation (rebuilt)    ✓  deviation/ratio signals
  NB2: Prophet forecasting           ✓  20 models trained         # UPDATED — was 20
  NB3: Indicator scoring             ✓  boss's thresholds applied
  NB4: Final risk score              ✓  weighted composite
  NB5: Validation + metrics          ✓  residual analysis

FINAL RISK SCORES (Jul 2026 Current):

  Wheat   : 32.9 → Watch
  Corn    : 34.7 → Watch
  Rice    : 35.6 → Watch
  Soybean : 30.2 → Watch
  Barley  : 18.1 → Low

FORECAST (Aug-Oct 2026):

  Wheat   : 23.9 | 23.7 | 23.6  (Low | Low | Low)
  Corn    : 28.9 | 29.5 | 30.1  (Watch | Watch | Watch)
  Rice    : 29.9 | 29.9 | 30.1  (Watch | Watch | Watch)
  Soybean : 27.0 | 27.5 | 28.0  (Wa